In [1]:
"""
Модуль анализа объёмов и потока сделок (order flow analysis).

Состоит из трёх частей:
  1. build_volume_profile      — горизонтальные объёмы по уровням цены (Volume Profile),
                                  POC, Value Area, HVN/LVN — для использования в качестве
                                  уровней поддержки/сопротивления при прогнозе цены.
  2. classify_with_open_interest / classify_tape_without_oi
                                — оценка того, была ли сделка (или группа сделок)
                                  открытием новой позиции или закрытием существующей.
  3. detect_large_trades / large_player_footprint
                                — выделение крупных участников и оценка того,
                                  накапливают они позицию или распределяют (закрывают).

ВАЖНОЕ МЕТОДОЛОГИЧЕСКОЕ ЗАМЕЧАНИЕ
----------------------------------
Обезличенная лента сделок не содержит информации о позициях участников.
Поэтому:
  - Если у инструмента есть открытый интерес (OI) — фьючерсы, опционы — классификация
    open/close становится ПОЧТИ точной на уровне агрегатов (не отдельных сделок),
    т.к. изменение OI напрямую говорит о нетто-открытии/закрытии контрактов.
  - Если OI нет (спот, акции без деривативов) — можно только оценивать ВЕРОЯТНОСТЬ
    открытия/закрытия по косвенным признакам (агрессор, реакция цены, объёмный профиль,
    повторяемость сделок). Это эвристика, а не факт. Всегда возвращается confidence,
    а не бинарный ответ.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Literal


In [8]:

# ============================================================================
# 1. ГОРИЗОНТАЛЬНЫЕ ОБЪЁМЫ (VOLUME PROFILE)
# ============================================================================

@dataclass
class VolumeProfileResult:
    profile: pd.DataFrame          # index: price_level; columns: volume, buy_volume, sell_volume, delta
    poc_price: float                # Point of Control — уровень с максимальным объёмом
    value_area_high: float
    value_area_low: float
    hvn_levels: List[float]         # High Volume Nodes — локальные максимумы объёма (магниты цены)
    lvn_levels: List[float]         # Low Volume Nodes — локальные минимумы (зоны быстрого прохода)
    total_volume: float


def build_volume_profile(
    trades: pd.DataFrame,
    price_col: str = "price",
    volume_col: str = "volume",
    side_col: Optional[str] = "side",     # значения 'buy'/'sell' — сторона агрессора, если известна
    bin_size: float = 0.01,
    value_area_pct: float = 0.70,
    time_col: Optional[str] = None,
    halflife: Optional[float] = None,     # период полу-затухания веса (в тех же единицах, что time_col)
) -> VolumeProfileResult:
    """
    Строит горизонтальный профиль объёма.

    halflife позволяет делать recency-weighted профиль: старые сделки учитываются
    с меньшим весом. Это важно для прогноза цены — актуальная структура ликвидности
    важнее исторической, которая могла быть "съедена" рынком.
    """
    df = trades.copy()

    # веса по времени (экспоненциальное затухание)
    if time_col is not None and halflife is not None and halflife > 0:
        t = df[time_col].astype(float)
        t_max = t.max()
        weight = 0.5 ** ((t_max - t) / halflife)
    else:
        weight = 1.0

    df["_w_volume"] = df[volume_col] * weight
    df["_bin"] = (df[price_col] / bin_size).round() * bin_size

    if side_col is not None and side_col in df.columns:
        buy_mask = df[side_col] == "buy"
        sell_mask = df[side_col] == "sell"
        grouped = df.groupby("_bin").agg(
            volume=("_w_volume", "sum"),
        )
        grouped["buy_volume"] = df[buy_mask].groupby("_bin")["_w_volume"].sum()
        grouped["sell_volume"] = df[sell_mask].groupby("_bin")["_w_volume"].sum()
        grouped[["buy_volume", "sell_volume"]] = grouped[["buy_volume", "sell_volume"]].fillna(0.0)
        grouped["delta"] = grouped["buy_volume"] - grouped["sell_volume"]
    else:
        grouped = df.groupby("_bin").agg(volume=("_w_volume", "sum"))
        grouped["buy_volume"] = np.nan
        grouped["sell_volume"] = np.nan
        grouped["delta"] = np.nan

    grouped = grouped.sort_index()
    total_volume = grouped["volume"].sum()

    # --- POC ---
    poc_price = grouped["volume"].idxmax()

    # --- Value Area: классический алгоритм "расширение от POC" ---
    levels = grouped.index.to_list()
    poc_idx = levels.index(poc_price)
    included = {poc_idx}
    accumulated = grouped["volume"].iloc[poc_idx]
    target = total_volume * value_area_pct

    lo, hi = poc_idx, poc_idx
    while accumulated < target and (lo > 0 or hi < len(levels) - 1):
        # смотрим, какая сторона (выше/ниже) даёт больший объём на следующем шаге
        vol_down = grouped["volume"].iloc[lo - 1] if lo > 0 else -1
        vol_up = grouped["volume"].iloc[hi + 1] if hi < len(levels) - 1 else -1
        if vol_down >= vol_up:
            lo -= 1
            accumulated += grouped["volume"].iloc[lo]
        else:
            hi += 1
            accumulated += grouped["volume"].iloc[hi]

    value_area_low = levels[lo]
    value_area_high = levels[hi]

    # --- HVN / LVN: локальные экстремумы объёма (простое peak-detection) ---
    vol_arr = grouped["volume"].to_numpy()
    hvn_levels, lvn_levels = [], []
    for i in range(1, len(vol_arr) - 1):
        if vol_arr[i] > vol_arr[i - 1] and vol_arr[i] > vol_arr[i + 1]:
            hvn_levels.append(levels[i])
        elif vol_arr[i] < vol_arr[i - 1] and vol_arr[i] < vol_arr[i + 1]:
            lvn_levels.append(levels[i])

    return VolumeProfileResult(
        profile=grouped.drop(columns=[c for c in ["_bin"] if c in grouped.columns]),
        poc_price=poc_price,
        value_area_high=value_area_high,
        value_area_low=value_area_low,
        hvn_levels=hvn_levels,
        lvn_levels=lvn_levels,
        total_volume=total_volume,
    )


def nearest_levels(vp: VolumeProfileResult, current_price: float, n: int = 3) -> pd.DataFrame:
    """
    Возвращает n ближайших HVN (потенциальные поддержка/сопротивление, "магниты")
    и LVN (зоны, где цена исторически двигалась быстро — вероятные зоны быстрого прохода)
    относительно текущей цены. Используется как входной признак для модели прогноза цены.
    """
    rows = []
    for lvl in vp.hvn_levels:
        rows.append({"level": lvl, "type": "HVN", "distance": lvl - current_price})
    for lvl in vp.lvn_levels:
        rows.append({"level": lvl, "type": "LVN", "distance": lvl - current_price})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["abs_distance"] = out["distance"].abs()
    return out.sort_values("abs_distance").head(n).drop(columns="abs_distance")


In [9]:
# ============================================================================
# 2. КЛАССИФИКАЦИЯ СДЕЛОК: ОТКРЫТИЕ vs ЗАКРЫТИЕ ПОЗИЦИИ
# ============================================================================

@dataclass
class OpenCloseEstimate:
    label: str            # человекочитаемая интерпретация
    confidence: float      # 0..1 — насколько уверенно можно это утверждать
    direction: Literal["long_open", "long_close", "short_open", "short_close", "mixed"]


def classify_with_open_interest(
    period_df: pd.DataFrame,
    price_col: str = "price",
    oi_col: str = "open_interest",
    buy_vol_col: str = "buy_volume",
    sell_vol_col: str = "sell_volume",
) -> pd.DataFrame:
    """
    Наиболее надёжный путь — доступен для инструментов с открытым интересом
    (фьючерсы, опционы). period_df — агрегация по интервалам времени (например, 1 мин/1 час),
    с колонками: цена (последняя в периоде), open_interest (значение на конец периода),
    buy_volume/sell_volume (объём на стороне агрессора-покупателя/продавца за период).

    Логика:
      ΔOI > 0 & Δprice > 0  -> открытие новых длинных позиций (бычье, "новые деньги")
      ΔOI > 0 & Δprice < 0  -> открытие новых коротких позиций (медвежье, "новые деньги")
      ΔOI < 0 & Δprice > 0  -> закрытие коротких (шорт-ковер) — рост на закрытии,
                               движение менее устойчиво, чем открытие лонгов
      ΔOI < 0 & Δprice < 0  -> закрытие длинных (ликвидация/фиксация) — падение на закрытии
      |ΔOI| мал относительно объёма -> просто переток позиций между участниками
                               (одни открывают, другие закрывают в похожем объёме)
    """
    df = period_df.copy()
    df["d_price"] = df[price_col].diff()
    df["d_oi"] = df[oi_col].diff()
    df["total_volume"] = df[buy_vol_col] + df[sell_vol_col]

    # доля изменения OI относительно суммарного объёма — мера "чистоты" сигнала
    with np.errstate(divide="ignore", invalid="ignore"):
        df["oi_change_ratio"] = (df["d_oi"].abs() / df["total_volume"]).replace([np.inf, -np.inf], np.nan)

    def _row_label(row):
        if pd.isna(row["d_oi"]) or pd.isna(row["d_price"]):
            return "недостаточно данных", 0.0, "mixed"
        ratio = row["oi_change_ratio"] if not pd.isna(row["oi_change_ratio"]) else 0.0
        confidence = float(np.clip(ratio, 0.0, 1.0))

        if row["d_oi"] > 0 and row["d_price"] > 0:
            return "открытие новых лонгов (новые покупатели)", confidence, "long_open"
        if row["d_oi"] > 0 and row["d_price"] < 0:
            return "открытие новых шортов (новые продавцы)", confidence, "short_open"
        if row["d_oi"] < 0 and row["d_price"] > 0:
            return "закрытие шортов (шорт-ковер)", confidence, "short_close"
        if row["d_oi"] < 0 and row["d_price"] < 0:
            return "закрытие лонгов (фиксация/ликвидация)", confidence, "long_close"
        return "переток позиций между участниками", 1.0 - confidence, "mixed"

    labels = df.apply(_row_label, axis=1, result_type="expand")
    df[["oc_label", "oc_confidence", "oc_direction"]] = labels
    return df


def classify_tape_without_oi(
    trades: pd.DataFrame,
    price_col: str = "price",
    volume_col: str = "volume",
    time_col: str = "time",
    side_col: Optional[str] = None,          # если None — сторона агрессора определяется tick-rule
    vp: Optional[VolumeProfileResult] = None, # объёмный профиль для контекста (см. build_volume_profile)
    lookahead_n: int = 20,                    # сколько сделок вперёд смотреть для оценки "устойчивости" движения
    large_quantile: float = 0.9,
) -> pd.DataFrame:
    """
    Эвристическая оценка open/close БЕЗ данных по открытому интересу — то есть по чистому тейпу.
    Возвращает по каждой сделке confidence-скор "вероятнее открытие" vs "вероятнее закрытие".

    Это НЕ детерминированный ответ, а сумма косвенных признаков:

      a) Сторона агрессора (tick rule, если side не дан явно):
         цена сделки выше предыдущей -> агрессор покупатель, ниже -> продавец.

      b) Аномальный размер (крупная сделка относительно скользящего окна) —
         крупные сделки чаще отражают решения институциональных игроков.

      c) Устойчивость движения после сделки (price drift): если после крупной покупки
         цена продолжает идти вверх без быстрого возврата — это больше похоже на
         ОТКРЫТИЕ новой позиции (агрессивный вход, есть кому продолжать покупать).
         Если цена быстро возвращается назад — это больше похоже на ЗАКРЫТИЕ/поглощение
         (крупный ордер был "съеден" встречной ликвидностью, нового направленного
         интереса не возникло).

      d) Контекст объёмного профиля: сделка на LVN (зона низкого исторического объёма) —
         скорее прорыв/новое позиционирование (открытие). Сделка на HVN (зона, где рынок
         уже много торговал) — скорее фиксация/ребалансировка существующих участников
         (закрытие или обмен позициями между старыми игроками).

    Итоговый score считается как взвешенная сумма нормированных признаков в диапазоне [-1, 1]:
      -1..0   -> сильнее похоже на закрытие
       0..+1  -> сильнее похоже на открытие
    """
    df = trades.sort_values(time_col).reset_index(drop=True).copy()

    # (a) сторона агрессора
    if side_col is not None and side_col in df.columns:
        df["aggressor"] = df[side_col]
    else:
        prev_price = df[price_col].shift(1)
        df["aggressor"] = np.where(df[price_col] > prev_price, "buy",
                            np.where(df[price_col] < prev_price, "sell", "unknown"))

    # (b) относительный размер сделки (z-score в скользящем окне)
    roll_mean = df[volume_col].rolling(200, min_periods=20).mean()
    roll_std = df[volume_col].rolling(200, min_periods=20).std().replace(0, np.nan)
    df["size_z"] = (df[volume_col] - roll_mean) / roll_std
    df["size_z"] = df["size_z"].fillna(0.0)
    large_threshold = df[volume_col].quantile(large_quantile)
    df["is_large"] = df[volume_col] >= large_threshold

    # (c) устойчивость движения после сделки (drift)
    future_price = df[price_col].shift(-lookahead_n)
    price_after = future_price - df[price_col]
    # нормируем знак drift относительно направления агрессора: совпадение направления = продолжение (открытие)
    sign_aggr = np.where(df["aggressor"] == "buy", 1, np.where(df["aggressor"] == "sell", -1, 0))
    drift_alignment = np.sign(price_after.fillna(0)) * sign_aggr  # +1 продолжение, -1 разворот
    df["drift_alignment"] = drift_alignment

    # (d) контекст объёмного профиля
    if vp is not None:
        bin_size_guess = np.median(np.diff(sorted(vp.profile.index.to_list()))) if len(vp.profile) > 1 else 0.01
        def _hvn_lvn_score(p):
            if not vp.hvn_levels and not vp.lvn_levels:
                return 0.0
            nearest_hvn = min((abs(p - l) for l in vp.hvn_levels), default=np.inf)
            nearest_lvn = min((abs(p - l) for l in vp.lvn_levels), default=np.inf)
            if nearest_hvn < nearest_lvn:
                return -0.5   # ближе к HVN -> смещаем к "закрытие/ребаланс"
            elif nearest_lvn < nearest_hvn:
                return 0.5    # ближе к LVN -> смещаем к "открытие/прорыв"
            return 0.0
        df["profile_context"] = df[price_col].apply(_hvn_lvn_score)
    else:
        df["profile_context"] = 0.0

    # --- итоговый взвешенный скор ---
    w_size = 0.25
    w_drift = 0.5
    w_profile = 0.25

    size_component = np.tanh(df["size_z"].clip(lower=0) / 3.0) * w_size  # только усиливает уверенность крупных сделок
    drift_component = df["drift_alignment"].fillna(0) * w_drift
    profile_component = df["profile_context"] * w_profile

    df["open_close_score"] = size_component + drift_component + profile_component
    df["open_close_score"] = df["open_close_score"].clip(-1, 1)

    df["interpretation"] = np.where(
        df["open_close_score"] > 0.15, "вероятнее открытие позиции",
        np.where(df["open_close_score"] < -0.15, "вероятнее закрытие позиции", "неопределённо")
    )

    return df[[time_col, price_col, volume_col, "aggressor", "is_large", "size_z",
               "drift_alignment", "profile_context", "open_close_score", "interpretation"]]


In [10]:

# ============================================================================
# 3. ПОВЕДЕНИЕ КРУПНЫХ ИГРОКОВ
# ============================================================================

def detect_large_trades(
    trades: pd.DataFrame,
    volume_col: str = "volume",
    quantile: float = 0.95,
) -> pd.DataFrame:
    """Помечает сделки, превышающие заданный квантиль объёма, как крупные."""
    df = trades.copy()
    threshold = df[volume_col].quantile(quantile)
    df["is_large"] = df[volume_col] >= threshold
    df["large_threshold"] = threshold
    return df


def large_player_footprint(
    large_trades_classified: pd.DataFrame,
    time_col: str = "time",
    volume_col: str = "volume",
    aggressor_col: str = "aggressor",
    window: str = "1h",
) -> pd.DataFrame:
    """
    Строит скользящий нетто-дельта крупных сделок (крупные покупки минус крупные продажи)
    и его накопленную сумму.

    Интерпретация для прогноза:
      - Устойчивый рост накопленной дельты при плоской/падающей цене -> вероятная тихая
        АККУМУЛЯЦИЯ (крупный игрок набирает позицию, не двигая цену сильно — часто
        предшествует движению вверх).
      - Устойчивое падение накопленной дельты при плоской/растущей цене -> вероятная
        тихая ДИСТРИБУЦИЯ (крупный игрок распродаёт в силу — часто предшествует падению).
    Также детектируется "iceberg"-паттерн: несколько сделок близкого размера по одной
    цене в узком временном окне — признак алгоритмического дробления крупного ордера.
    """
    df = large_trades_classified.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.set_index(time_col)

    signed_volume = np.where(df[aggressor_col] == "buy", df[volume_col],
                       np.where(df[aggressor_col] == "sell", -df[volume_col], 0))
    df["signed_large_volume"] = signed_volume

    footprint = df["signed_large_volume"].resample(window).sum().to_frame("net_large_delta")
    footprint["cumulative_large_delta"] = footprint["net_large_delta"].cumsum()
    return footprint


def detect_iceberg_pattern(
    trades: pd.DataFrame,
    price_col: str = "price",
    volume_col: str = "volume",
    time_col: str = "time",
    time_window_seconds: float = 30.0,
    size_tolerance_pct: float = 0.15,
    min_repeats: int = 4,
) -> pd.DataFrame:
    """
    Ищет группы сделок по одинаковой цене, близкого размера, идущих плотно во времени —
    типичный след дробления крупного ордера (iceberg / алго-исполнение).
    Возвращает сгруппированные кластеры с суммарным объёмом.
    """
    df = trades.sort_values(time_col).copy()
    df[time_col] = pd.to_datetime(df[time_col])

    clusters = []
    current_cluster = []

    for _, row in df.iterrows():
        if not current_cluster:
            current_cluster = [row]
            continue
        last = current_cluster[-1]
        same_price = abs(row[price_col] - last[price_col]) < 1e-9
        close_in_time = (row[time_col] - last[time_col]).total_seconds() <= time_window_seconds
        similar_size = abs(row[volume_col] - last[volume_col]) / max(last[volume_col], 1e-9) <= size_tolerance_pct
        if same_price and close_in_time and similar_size:
            current_cluster.append(row)
        else:
            if len(current_cluster) >= min_repeats:
                clusters.append(current_cluster)
            current_cluster = [row]
    if len(current_cluster) >= min_repeats:
        clusters.append(current_cluster)

    rows = []
    for cl in clusters:
        rows.append({
            "start_time": cl[0][time_col],
            "end_time": cl[-1][time_col],
            "price": cl[0][price_col],
            "num_trades": len(cl),
            "total_volume": sum(r[volume_col] for r in cl),
        })
    return pd.DataFrame(rows)

In [11]:

# ============================================================================
# ОРКЕСТРАЦИЯ: единый вызов для отчёта по инструменту
# ============================================================================

def analyze_instrument(
    trades: pd.DataFrame,
    price_col: str = "price",
    volume_col: str = "volume",
    time_col: str = "time",
    side_col: Optional[str] = None,
    oi_period_df: Optional[pd.DataFrame] = None,
    bin_size: float = 0.01,
) -> dict:
    """Собирает всё вместе: профиль объёма, классификацию open/close, крупных игроков."""
    vp = build_volume_profile(
        trades, price_col=price_col, volume_col=volume_col,
        side_col=side_col, bin_size=bin_size,
        time_col=time_col, halflife=None,
    )

    classified = classify_tape_without_oi(
        trades, price_col=price_col, volume_col=volume_col,
        time_col=time_col, side_col=side_col, vp=vp,
    )

    large = detect_large_trades(trades, volume_col=volume_col)
    large_classified = classified.merge(
        large[[time_col, "is_large"]], on=time_col, suffixes=("", "_dup")
    )
    large_only = large_classified[large_classified["is_large"]]

    footprint = large_player_footprint(large_only, time_col=time_col, volume_col=volume_col)
    icebergs = detect_iceberg_pattern(trades, price_col=price_col, volume_col=volume_col, time_col=time_col)

    result = {
        "volume_profile": vp,
        "trade_classification": classified,
        "large_player_footprint": footprint,
        "iceberg_clusters": icebergs,
    }

    if oi_period_df is not None:
        result["oi_based_classification"] = classify_with_open_interest(oi_period_df, price_col=price_col)

    return result

In [21]:

# ============================================================================
# ПРИМЕР ИСПОЛЬЗОВАНИЯ (синтетические данные)
# ============================================================================

if __name__ == "__main__":
#     df = pd.read_parquet(r"D:\sd\moexalgo\data\lkoh\orderflow_lkoh_1s.parquet")
    
    report = analyze_instrument(df_test)

    print("POC:", report["volume_profile"].poc_price)
    print("Value Area:", report["volume_profile"].value_area_low, "-", report["volume_profile"].value_area_high)
    print("HVN levels (первые 5):", report["volume_profile"].hvn_levels[:5])
    print("\nПримеры классификации сделок:")
    print(report["trade_classification"].tail(10))
    print("\nFootprint крупных игроков (последние строки):")
    print(report["large_player_footprint"].tail())
    print("\nОбнаруженные iceberg-кластеры:")
    print(report["iceberg_clusters"])

POC: 5505.0
Value Area: 5479.0 - 5573.5
HVN levels (первые 5): [5442.5, 5443.5, 5445.0, 5446.0, 5447.0]

Примеры классификации сделок:
                           time   price  volume aggressor  is_large    size_z  \
65597 2026-04-06 20:48:53+00:00  5573.0       1   unknown     False -0.348547   
65598 2026-04-06 20:48:58+00:00  5574.5       1       buy     False -0.347145   
65599 2026-04-06 20:49:00+00:00  5573.0       7      sell     False -0.291390   
65600 2026-04-06 20:49:07+00:00  5574.0      69       buy     False  0.282837   
65601 2026-04-06 20:49:16+00:00  5575.0       1       buy     False -0.350241   
65602 2026-04-06 20:49:21+00:00  5575.5       2       buy     False -0.340983   
65603 2026-04-06 20:49:34+00:00  5576.5     120       buy      True  0.751406   
65604 2026-04-06 20:49:44+00:00  5575.5       5      sell     False -0.318082   
65605 2026-04-06 20:49:48+00:00  5576.5       1       buy     False -0.354611   
65606 2026-04-06 20:49:49+00:00  5576.5       2   unkno

In [26]:
# report["trade_classification"].to_csv('trade_classification_lkoh_1809_1.csv', encoding='cp1251')
# report["large_player_footprint"].to_csv('large_players_lkoh_1809.csv', encoding='cp1251')
report["iceberg_clusters"].to_csv('iceberg_clusters_lkoh_1809.csv', encoding='cp1251')

In [6]:
df = pd.read_parquet(r"D:\sd\moexalgo\data\lkoh\orderflow_lkoh_1s.parquet")
print(df.columns)
print(df.head())

Index(['TICKER_CC', 'time', 'price', 'volume', 'n_trades', 'signed_vol',
       'buy_trades', 'notional_sum', 'buy_volume', 'sell_volume', 'vwap',
       'buy_ratio', 'trade_ratio', 'ofi', 'avg_trade_sz', 'source'],
      dtype='str')
   TICKER_CC                      time   price  volume  n_trades  signed_vol  \
0  LKOH_TQBR 2024-01-03 06:59:51+00:00  6774.0    2378       153         332   
1  LKOH_TQBR 2024-01-03 07:00:00+00:00  6770.0      25         5         -25   
2  LKOH_TQBR 2024-01-03 07:00:01+00:00  6767.0     228        24         -88   
3  LKOH_TQBR 2024-01-03 07:00:02+00:00  6760.0     143        15         -77   
4  LKOH_TQBR 2024-01-03 07:00:03+00:00  6758.5      93        10          63   

   buy_trades  notional_sum  buy_volume  sell_volume         vwap  buy_ratio  \
0          88    16108572.0      1355.0       1023.0  6774.000000   0.569807   
1           0      169250.0         0.0         25.0  6770.000000   0.000000   
2           7     1541807.0        70.0     

In [20]:
df_test = df.query("TICKER_CC == 'LKOH_TQBR' and time > '2026-03-31 06:59:51+00:00'")
print(df_test.head())

print(len(df_test))

         TICKER_CC                      time   price  volume  n_trades  \
9710861  LKOH_TQBR 2026-03-31 07:00:00+00:00  5682.5       1         1   
9710862  LKOH_TQBR 2026-03-31 07:00:02+00:00  5682.5       1         1   
9710863  LKOH_TQBR 2026-03-31 07:00:03+00:00  5682.5      30         2   
9710864  LKOH_TQBR 2026-03-31 07:00:05+00:00  5683.0       1         1   
9710865  LKOH_TQBR 2026-03-31 07:00:06+00:00  5682.5       1         1   

         signed_vol  buy_trades  notional_sum  buy_volume  sell_volume  \
9710861          -1           0        5682.5         0.0          1.0   
9710862          -1           0        5682.5         0.0          1.0   
9710863          28           1      170489.5        29.0          1.0   
9710864          -1           0        5683.0         0.0          1.0   
9710865          -1           0        5682.5         0.0          1.0   

                vwap  buy_ratio  trade_ratio  ofi  avg_trade_sz    source  
9710861  5682.500000   0.000000   

# Только скрипт по объёмам

In [30]:
df_large = df.query("TICKER_CC == 'LKOH_TQBR' and time > '2025-12-31 06:59:51+00:00'")
print(df_large.head())

print(len(df_large))
print(df_large.columns)

         TICKER_CC                      time   price  volume  n_trades  \
8753224  LKOH_TQBR 2025-12-31 07:00:04+00:00  5906.0       1         1   
8753225  LKOH_TQBR 2025-12-31 07:00:58+00:00  5911.0       1         1   
8753226  LKOH_TQBR 2025-12-31 07:01:55+00:00  5908.5      17         2   
8753227  LKOH_TQBR 2025-12-31 07:02:05+00:00  5911.5      17         1   
8753228  LKOH_TQBR 2025-12-31 07:02:35+00:00  5911.5      22         1   

         signed_vol  buy_trades  notional_sum  buy_volume  sell_volume  \
8753224          -1           0        5906.0         0.0          1.0   
8753225           1           1        5911.0         1.0          0.0   
8753226         -17           0      100426.5         0.0         17.0   
8753227          17           1      100495.5        17.0          0.0   
8753228          22           1      130053.0        22.0          0.0   

                vwap  buy_ratio  trade_ratio  ofi  avg_trade_sz    source  
8753224  5906.000000        0.0   

In [32]:
 
def large_player_footprint(
    bars: pd.DataFrame,
    ticker_col: str = "TICKER_CC",
    time_col: str = "time",
    volume_col: str = "volume",
    buy_volume_col: str = "buy_volume",
    sell_volume_col: str = "sell_volume",
    signed_vol_col: str = "signed_vol",
    n_trades_col: str = "n_trades",
    buy_trades_col: str = "buy_trades",
    notional_col: str = "notional_sum",
    window: str = "1h",
    min_avg_trade_size: Optional[float] = None,
    min_notional: Optional[float] = None,
) -> pd.DataFrame:
    """
    Строит скользящий нетто-дельта (крупные покупки минус крупные продажи) и его накопленную
    сумму — принимает УЖЕ агрегированные данные (бары), без какой-либо предварительной обработки:
    ожидаемый набор колонок — 'TICKER_CC', 'time', 'price', 'volume', 'n_trades', 'signed_vol',
    'buy_trades', 'notional_sum', 'buy_volume', 'sell_volume', 'vwap', 'buy_ratio', 'trade_ratio',
    'ofi', 'avg_trade_sz', 'source'. Сторону сделки пересчитывать не нужно — signed_vol,
    buy_volume/sell_volume уже посчитаны в источнике.
 
    Поддерживает несколько инструментов одновременно: если в данных есть ticker_col
    (по умолчанию 'TICKER_CC'), агрегация и накопленная сумма считаются ОТДЕЛЬНО по каждому
    тикеру. Если колонки нет — вся выборка считается одним инструментом.
 
    ОТБОР "КРУПНЫХ" БАРОВ (опционально). Если входные данные — это все бары без предварительной
    фильтрации по размеру, можно отобрать бары, где явно доминируют крупные сделки, прямо здесь:
      - min_avg_trade_size — минимальный средний размер сделки в баре (avg_trade_sz);
      - min_notional       — минимальный оборот бара (notional_sum).
    Если фильтрация уже сделана заранее (входные данные и так только "крупные" бары/сделки) —
    оба параметра можно оставить None, тогда используются все переданные строки.
 
    ПЕРЕСЧЁТ ПРОИЗВОДНЫХ МЕТРИК ПРИ АГРЕГАЦИИ. vwap, buy_ratio, trade_ratio и avg_trade_sz
    нельзя просто усреднять по барам (это взвешенные величины) — они пересчитываются заново
    из просуммированных за окно notional_sum/volume/buy_volume/n_trades/buy_trades, чтобы
    результат был корректным, а не средним от средних.
 
    Интерпретация для прогноза:
      - Устойчивый рост накопленной дельты при плоской/падающей цене -> вероятная тихая
        АККУМУЛЯЦИЯ (крупный игрок набирает позицию, не двигая цену сильно — часто
        предшествует движению вверх).
      - Устойчивое падение накопленной дельты при плоской/растущей цене -> вероятная
        тихая ДИСТРИБУЦИЯ (крупный игрок распродаёт в силу — часто предшествует падению).
    """
    df = bars.copy()
    df[time_col] = pd.to_datetime(df[time_col])
 
    # --- fallback, если каких-то производных колонок нет в источнике ---
    if signed_vol_col not in df.columns:
        df[signed_vol_col] = df[buy_volume_col] - df[sell_volume_col]
    if sell_volume_col not in df.columns:
        df[sell_volume_col] = df[volume_col] - df[buy_volume_col]
 
    # --- опциональный отбор баров, где явно доминируют крупные сделки ---
    if min_avg_trade_size is not None and "avg_trade_sz" in df.columns:
        df = df[df["avg_trade_sz"] >= min_avg_trade_size]
    if min_notional is not None and notional_col in df.columns:
        df = df[df[notional_col] >= min_notional]
 
    if df.empty:
        return pd.DataFrame(columns=[
            time_col, "volume", "buy_volume", "sell_volume", "net_large_delta",
            "n_trades", "buy_trades", "notional_sum", "vwap", "buy_ratio",
            "trade_ratio", "avg_trade_sz", "cumulative_large_delta",
        ])
 
    has_ticker = ticker_col in df.columns
    df = df.set_index(time_col)
 
    group_cols = [ticker_col, pd.Grouper(freq=window)] if has_ticker else [pd.Grouper(freq=window)]
    grouped = df.groupby(group_cols)
 
    footprint = grouped.agg(**{
        "volume": (volume_col, "sum"),
        "buy_volume": (buy_volume_col, "sum"),
        "sell_volume": (sell_volume_col, "sum"),
        "net_large_delta": (signed_vol_col, "sum"),
        "n_trades": (n_trades_col, "sum"),
        "buy_trades": (buy_trades_col, "sum"),
        "notional_sum": (notional_col, "sum"),
    })
 
    # производные метрики — пересчитаны из сумм за окно, а не усреднены по барам
    footprint["vwap"] = footprint["notional_sum"] / footprint["volume"].replace(0, np.nan)
    footprint["buy_ratio"] = footprint["buy_volume"] / footprint["volume"].replace(0, np.nan)
    footprint["trade_ratio"] = footprint["buy_trades"] / footprint["n_trades"].replace(0, np.nan)
    footprint["avg_trade_sz"] = footprint["volume"] / footprint["n_trades"].replace(0, np.nan)
 
    if has_ticker:
        footprint["cumulative_large_delta"] = footprint.groupby(level=ticker_col)["net_large_delta"].cumsum()
    else:
        footprint["cumulative_large_delta"] = footprint["net_large_delta"].cumsum()
 
    return footprint.reset_index()
 
 

In [34]:
large_player_footprint(df_large).to_csv('footprint_lkoh_1909.csv', encoding='cp1251')

In [ ]:
df_large.to_csv('iceberg_clusters_lkoh_1809.csv', encoding='cp1251')